In [158]:
import pandas as pd
housing = pd.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing.csv')

# import polars as pl
# housing = pl.read_csv('https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/housing.csv')

In [159]:
housing.head()

,id,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,...,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price
0,1565930130,20141104T000000,4,3.25,3760,4675,2.0,0,0,3,...,2740,1020,2007,0,98038,47.3862,-122.048,3280,4033,429900.0
1,3279000420,20150115T000000,3,1.75,1460,7800,1.0,0,0,2,...,1040,420,1979,0,98023,47.3035,-122.382,1310,7865,233000.0
2,194000575,20141014T000000,4,1.00,1340,5800,1.5,0,2,3,...,1340,0,1914,0,98116,47.5658,-122.389,1900,5800,455000.0
3,2115510160,20141208T000000,3,1.75,1440,8050,1.0,0,0,3,...,1440,0,1985,0,98023,47.3187,-122.390,1790,7488,258950.0
4,7522500005,20140815T000000,2,1.50,1780,4750,1.0,0,0,4,...,1080,700,1947,0,98117,47.6859,-122.395,1690,5962,555000.0


## Data Processing: Identifying Unnecessary Columns

To perform effective data processing, let's first examine the columns in our `housing` DataFrame. We'll look for columns that might be redundant, have too many unique values (high cardinality) for direct use, or are otherwise not relevant for predicting house prices.

In [160]:
print(housing.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             20000 non-null  int64  
 1   date           20000 non-null  object 
 2   bedrooms       20000 non-null  int64  
 3   bathrooms      20000 non-null  float64
 4   sqft_living    20000 non-null  int64  
 5   sqft_lot       20000 non-null  int64  
 6   floors         20000 non-null  float64
 7   waterfront     20000 non-null  int64  
 8   view           20000 non-null  int64  
 9   condition      20000 non-null  int64  
 10  grade          20000 non-null  int64  
 11  sqft_above     20000 non-null  int64  
 12  sqft_basement  20000 non-null  int64  
 13  yr_built       20000 non-null  int64  
 14  yr_renovated   20000 non-null  int64  
 15  zipcode        20000 non-null  int64  
 16  lat            20000 non-null  float64
 17  long           20000 non-null  float64
 18  sqft_l

In [161]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.cluster import KMeans

# 1. Feature Engineering
housing['date'] = pd.to_datetime(housing['date'])
min_date = housing['date'].min()
housing['date_numerical'] = (housing['date'] - min_date).dt.days / 365.25

In [162]:
from sklearn.preprocessing import MinMaxScaler
# Create an instance of the transform object we're going to use
scaler = MinMaxScaler()
housing['view_scaled'] = scaler.fit_transform(housing[['view']])
housing['condition_scaled'] = scaler.fit_transform(housing[['condition']])
housing['grade_scaled'] = scaler.fit_transform(housing[['grade']])
housing['sqft_living_scaled'] = scaler.fit_transform(housing[['sqft_living']])
housing['sqft_lot_scaled'] = scaler.fit_transform(housing[['sqft_lot']])

# A better luxury indicator: the percentage of these high-end attributes that are in the top 10% (scaled value > 0.9)
housing['luxury_value'] = np.mean(housing[['view_scaled', 'condition_scaled', 'grade_scaled', 'sqft_living_scaled', 'sqft_lot_scaled']])

In [163]:
from sklearn.cluster import KMeans

# 1. Create Neighborhood Clusters
kmeans = KMeans(n_clusters=50, random_state=42, n_init=10)
housing['neighborhood_cluster'] = kmeans.fit_predict(housing[['lat', 'long']])


# 3. Distance to Seattle Center
seattle_center = (47.6062, -122.3321)
housing['dist_to_center'] = np.sqrt((housing['lat'] - seattle_center[0])**2 + (housing['long'] - seattle_center[1])**2)

# 4. Property Age (Effective Age)
housing['house_age'] = 2015 - housing['yr_built']
housing['renovated_age'] = np.where(housing['yr_renovated'] > 0, 2015 - housing['yr_renovated'], housing['house_age'])

features = ['date_numerical', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'view',
            'condition', 'lat', 'long', 'sqft_living15', 'sqft_lot15', 'grade', 'luxury_value', 'dist_to_center', 'renovated_age']

X = housing[features]
y = housing['price']

In [164]:
# 2. Data Splitting - using log target for better distribution
y_log = np.log1p(y)
X_train, X_test, y_train_log, y_test_log = train_test_split(X, y_log, test_size=0.2, random_state=42)

# 3. Final Model Training
model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6, random_state=42)
model.fit(X_train, y_train_log)

# Evaluation (convert back from log space)
log_predictions = model.predict(X_test)
predictions = np.expm1(log_predictions)

In [166]:
# Compute the Root Mean Squared Error of the predictions
from sklearn.metrics import root_mean_squared_error

result = root_mean_squared_error(y_test_log, predictions)
print(f"Current RMSE: ${result:,.2f}")


Current RMSE: $634,012.05
